# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DipeshGhimire33/Flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [14]:
import pandas as pd
import numpy as np

df =pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

feature_cols = [ "days_since_last_update", "content_age_days", "impressions_90d", "clicks_90d", "sessions_90d", "engagement_rate", "trend_pct", "avg_position", "ctr" ]
X = df[feature_cols].copy()

for col in feature_cols:
    X[col] = X[col].fillna(X[col].median())
    
trend_map = { "down": 1, "stable": 0, "flat": 0, "up": -1, "new": 0 }
X["trend_signal"] = ( df["trend_direction"] .map(trend_map) .fillna(0) )

In [15]:
print("Feature shape:", X.shape) 
display(X.head())

Feature shape: (30000, 10)


,days_since_last_update,content_age_days,impressions_90d,clicks_90d,sessions_90d,engagement_rate,trend_pct,avg_position,ctr,trend_signal
0,20,187,3803,29,17,5.88,-41.4,10.6,0.76,1
1,25,445,15320,7,9,0.00,-57.7,20.3,0.05,1
2,20,141,12581,11,11,0.00,-60.9,36.5,0.09,1
3,22,463,11751,58,78,1.28,-13.8,6.2,0.49,0
4,14,263,19140,24,145,0.00,-34.7,44.0,0.13,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature definitions and timing

| Feature                  | What it means                           | Missing values           | Exists before scoring? |
| ------------------------ | --------------------------------------- | ------------------------ | ---------------------- |
| `days_since_last_update` | Days since the page was last updated    | Median                   | Yes                    |
| `content_age_days`       | Age of the content                      | Median                   | Yes                    |
| `impressions_90d`        | Search impressions over 90 days         | Median                   | Yes                    |
| `clicks_90d`             | Search clicks over 90 days              | Median                   | Yes                    |
| `sessions_90d`           | Sessions over 90 days                   | Median                   | Yes                    |
| `engagement_rate`        | Share of sessions that were engaged     | Median                   | Yes                    |
| `trend_pct`              | Recent percentage change in performance | Median                   | Yes                    |
| `avg_position`           | Average search position                 | Median                   | Yes                    |
| `ctr`                    | Click-through rate                      | Median                   | Yes                    |
| `trend_signal`           | Numeric version of trend direction      | Filled with 0 if missing | Yes                    |

All features are calculated from information already available in the dataset at the time of ranking. They are used for **decision-support scoring**, not for predicting a future observed outcome.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [17]:
label_like = [ col for col in df.columns if any(word in col.lower() for word in ["label", "target", "refresh", "outcome"]) ]
print("Potential label-derived columns:", label_like)

Potential label-derived columns: []


In [18]:
future_like = [ col for col in feature_cols
               if any(word in col.lower() for word in ["future", "next", "forecast"])
               ]
print("Potential future-window features:", future_like)

Potential future-window features: []


In [19]:
excluded = ["content_id", "client_id", "provider_used", "model_used"]
used_excluded = [col for col in X.columns if col in excluded]
print("Excluded fields accidentally used:", used_excluded)

Excluded fields accidentally used: []


Result

The initial feature set does not use a refresh label, future-labeled outcome, or client identifier. The ranking should therefore be treated as a current-data opportunity score, not a future-performance prediction.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## Fields excluded from scoring

* `content_id` — identifier only; does not describe refresh opportunity.
* `client_id` — identifies the client; could create client-specific bias.
* `provider_used` — describes the data/provider source, not page opportunity.
* `model_used` — describes the model used to generate content, not current refresh need.
* `search_volume` — describes keyword demand, not whether the existing page needs a refresh.
* `competition` — describes keyword competition rather than page condition.
* `competition_level` — categorical version of competition; not a core refresh signal.
* `cpc` — advertising cost signal; not directly related to content freshness.
* `content_type` — useful for segmentation, but excluded from the initial score to keep the ranking focused on page signals.
* `main_intent` — describes search intent, but is not a direct measure of refresh opportunity.
* `word_count` — length alone does not show whether content needs updating.
* `char_count` — same as word count; length alone is not sufficient evidence.
* `ai_sessions_90d` — AI traffic is not central to the initial refresh score.
* `ai_traffic_pct` — same reason; outside the core refresh signals.
* `scroll_events_90d` — engagement detail, but not necessary for the initial ranking.
* `days_with_impressions` — supporting availability signal, not a core refresh signal.
* `days_with_sessions` — supporting availability signal, not a core refresh signal.
* `age_tier` — categorical duplicate of content age.
* `age_tier_order` — derived version of `age_tier`.
* `freshness_tier` — categorical version of freshness already represented by `days_since_last_update`.
* `word_count_tier` — derived from word count, which is excluded.
* `char_count_tier` — derived from character count, which is excluded.
* `impression_tier` — derived from impressions, so it duplicates an included feature.
* `position_tier` — derived from `avg_position`, so it duplicates an included feature.


In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.